# Aula 4 — Um Sistema de Ponta a Ponta

Um classificador de cancelamento com regressão logística, validação cruzada estratificada e Optuna. A Parte A demonstra; na Parte B, você completa a implementação.


## Parte A: demonstração

Instale Optuna no Colab e carregue os 400 clientes.


In [ ]:
!pip -q install optuna
import pandas as pd
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

url = 'https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/clube_cafe_clientes.csv'
clientes = pd.read_csv(url)
print(clientes['cancelou'].value_counts())


### Entradas, treino e teste

A classe de cancelamento é menor (146 de 400). `stratify` preserva a proporção. O teste fica guardado até terminar o tuning.


In [ ]:
X = pd.get_dummies(clientes[['meses_de_casa', 'valor_mensal', 'entregas_atrasadas', 'plano']], dtype=int)
y = clientes['cancelou']
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
dobras = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


### Optuna escolhe os hiperparâmetros

`C` controla a regularização; `class_weight` testa se dar mais peso à classe menor ajuda. O F1 médio das cinco dobras orienta a busca. O `Pipeline` refaz a padronização dentro de cada dobra.


In [ ]:
def avaliar(trial):
    c = trial.suggest_float('C', 0.01, 10.0, log=True)
    peso = trial.suggest_categorical('class_weight', [None, 'balanced'])
    modelo = make_pipeline(StandardScaler(), LogisticRegression(C=c, class_weight=peso))
    return cross_val_score(modelo, X_treino, y_treino, cv=dobras, scoring='f1').mean()

estudo = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
estudo.optimize(avaliar, n_trials=15)
print(estudo.best_params, estudo.best_value)


### Avaliação final

Só agora o teste separado entra. Compare seu F1 nele com o F1 médio das dobras.


In [ ]:
modelo = make_pipeline(StandardScaler(), LogisticRegression(**estudo.best_params))
modelo.fit(X_treino, y_treino)
print(classification_report(y_teste, modelo.predict(X_teste)))


## Parte B: sua implementação

Complete os espaços e rode as células na ordem.


### Exercício 1: a classe majoritária

Calcule a fração de clientes que ficaram. Essa é a acurácia de quem sempre responde 'ficou'.


In [ ]:
# SEU CODIGO AQUI
fracao_que_ficou = ...
print(fracao_que_ficou)


### Exercício 2: teste outra métrica

Troque `scoring` por `average_precision` e compare os melhores hiperparâmetros. Use somente `X_treino` e `y_treino`.


In [ ]:
def minha_avaliacao(trial):
    c = trial.suggest_float('C', 0.01, 10.0, log=True)
    peso = trial.suggest_categorical('class_weight', [None, 'balanced'])
    modelo = make_pipeline(StandardScaler(), LogisticRegression(C=c, class_weight=peso))
    # SEU CODIGO AQUI
    return ...

meu_estudo = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
meu_estudo.optimize(minha_avaliacao, n_trials=15)
print(meu_estudo.best_params)


### Exercício 3: uma lista por limiar

Treine o modelo com os parâmetros do seu estudo e conte quantos clientes do teste entram na lista em 0,30 e em 0,50.


In [ ]:
meu_modelo = make_pipeline(StandardScaler(), LogisticRegression(**meu_estudo.best_params))
meu_modelo.fit(X_treino, y_treino)
riscos = meu_modelo.predict_proba(X_teste)[:, 1]
# SEU CODIGO AQUI
print('Limiar 0,30:', ...)
print('Limiar 0,50:', ...)
